In [1]:
import sys

sys.path.append('../../../')

In [2]:
from __future__ import annotations
from typing import Iterable, Optional, Union
from collections import OrderedDict

import copy
import inspect
import warnings

import torch
import torch.nn as nn

%load_ext autoreload
%autoreload 2

# from computer_vision.slowfast.mmaction.models.data_preprocessors.data_preprocessor import ActionDataPreprocessor
# from computer_vision.slowfast.mmaction.models.heads.slowfast_head import SlowFastHead
# from computer_vision.slowfast.mmaction.models.losses.cross_entropy_loss import CrossEntropyLoss
# from computer_vision.slowfast.mmaction.models.backbones.resnet3d_slowfast import ResNet3dSlowFast
# from computer_vision.slowfast.mmengine.model.utils import merge_dict
# from computer_vision.slowfast.mmaction.utils import ForwardResults
# from computer_vision.slowfast.mmengine.optim.optimizer.optimizer_wrapper import OptimWrapper
# from computer_vision.slowfast.mmengine.utils.misc import is_list_of
# from computer_vision.slowfast.mmengine.model.base_model.data_preprocessor import BaseDataPreprocessor

from mmengine import Config, DictAction
from computer_vision.slowfast.mmaction.utils import SampleList
from computer_vision.slowfast.mmaction.models.recognizers.recognizer3d import Recognizer3D

Originally, [BaseRecognizer](https://github.com/open-mmlab/mmaction2/blob/main/mmaction/models/recognizers/base.py) was derived from [BaseModel](https://github.com/open-mmlab/mmengine/blob/main/mmengine/model/base_model/base_model.py) which is inheritted from [BaseModule](https://github.com/open-mmlab/mmengine/blob/main/mmengine/model/base_module.py). But we do not need to those functions defined in the BaseModule and BaseModel. We want light and simple implementation

In [3]:
config_fpath='../config/slowfast_r50_8xb8-4x16x1-256e_kinetics400-rgb.py'
cfg=Config.fromfile(config_fpath)

recognizer=Recognizer3D(backbone=cfg.model.backbone, cls_head=cfg.model.cls_head, train_cfg=None, test_cfg=None,
                data_preprocessor=None)

In resnet3d_slowfast.ResNet3dSlowFast.__init__ slow_pathway={'type': 'resnet3d', 'depth': 50, 'pretrained': None, 'lateral': True, 'conv1_kernel': (1, 7, 7), 'dilations': (1, 1, 1, 1), 'conv1_stride_t': 1, 'pool1_stride_t': 1, 'inflate': (0, 0, 1, 1), 'norm_eval': False, 'speed_ratio': 8, 'channel_ratio': 8} 
In resnet3d_slowfast.ResNet3dSlowFast.__init__ fast_pathway={'type': 'resnet3d', 'depth': 50, 'pretrained': None, 'lateral': False, 'base_channels': 8, 'conv1_kernel': (5, 7, 7), 'conv1_stride_t': 1, 'pool1_stride_t': 1, 'norm_eval': False} 
In ResNet3dPathway._calculate_lateral_inplanes: depth=50, expansion=4, base_channels=64
stage 0 ----------
	planes=64, self.lateral=True, self.lateral_activate[i]=1, self.lateral_inv=False
stage 1 ----------
	planes=256, self.lateral=True, self.lateral_activate[i]=1, self.lateral_inv=False
stage 2 ----------
	planes=512, self.lateral=True, self.lateral_activate[i]=1, self.lateral_inv=False
stage 3 ----------
	planes=1024, self.lateral=True, se

In [6]:
inputs=torch.rand(3,3,32,180,180)
stage='head'
data_samples=None
test_mode=False

feats, predict_kwargs=recognizer.extract_feat(inputs, test_mode=True)
print('predict_kwargs ', predict_kwargs)
print('feats ', type(feats), [i.shape for i in feats])
predictions=recognizer.cls_head.predict(feats, data_samples, **predict_kwargs)
print('predictions ', type(predictions))

inputs.shape=torch.Size([3, 3, 32, 180, 180])
predict_kwargs  {}
feats  <class 'tuple'> [torch.Size([3, 2048, 4, 6, 6]), torch.Size([3, 256, 32, 6, 6])]


AttributeError: 'SlowFastHead' object has no attribute 'predict'